# Quick Colab Finetune: EfficientNet-B4 (RGB)

This notebook performs a short finetune on the Kaggle dataset `raedsaidi/deepfake` and saves checkpoints to `/content/checkpoints` and optionally Google Drive.

In [ ]:
from pathlib import Path

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Drive disabled; checkpoints stay in /content/checkpoints')

In [ ]:
!pip install -q kaggle timm albumentations

In [ ]:
import os
import shutil
from google.colab import files

USE_DRIVE_KAGGLE_JSON = True
KAGGLE_JSON_IN_DRIVE = Path('/content/drive/MyDrive/kaggle.json')

Path.home().joinpath('.kaggle').mkdir(parents=True, exist_ok=True)
target = Path.home().joinpath('.kaggle', 'kaggle.json')

if USE_DRIVE_KAGGLE_JSON and KAGGLE_JSON_IN_DRIVE.exists():
    shutil.copy(KAGGLE_JSON_IN_DRIVE, target)
    print('Copied kaggle.json from Drive')
else:
    print('Upload kaggle.json now')
    uploaded = files.upload()
    json_names = [name for name in uploaded.keys() if name.lower().endswith('.json')]
    if not json_names:
        raise RuntimeError('No JSON file uploaded')
    picked = json_names[0]
    with open(target, 'wb') as f:
        f.write(uploaded[picked])
    print('Using uploaded token file:', picked)

os.chmod(target, 0o600)
print('Kaggle API token ready at', target)

In [ ]:
!mkdir -p /content/data
!kaggle datasets download -d raedsaidi/deepfake -p /content/data --unzip
!find /content/data -maxdepth 4 -type d | head -n 30

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SUBSET_SIZE = 10000  # set to 'all' for full dataset
EPOCHS = 3
BATCH_SIZE = 32
LR = 1e-4
NUM_WORKERS = 2

DATA_ROOT = Path('/content/data/processed/FaceForensics++_C23')
if not DATA_ROOT.exists():
    candidates = list(Path('/content/data').rglob('FaceForensics++_C23'))
    if not candidates:
        raise FileNotFoundError('FaceForensics++_C23 not found under /content/data')
    DATA_ROOT = candidates[0]

print('Using DATA_ROOT =', DATA_ROOT)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

dataset = datasets.ImageFolder(str(DATA_ROOT), transform=transform)
print('Classes:', dataset.classes, 'Total:', len(dataset))

if SUBSET_SIZE == 'all' or SUBSET_SIZE >= len(dataset):
    train_ds = dataset
else:
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    train_ds = Subset(dataset, indices[:SUBSET_SIZE])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
print('Train batches:', len(train_loader))

In [ ]:
weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
backbone = models.efficientnet_b4(weights=weights)
backbone_body = nn.Sequential(*list(backbone.children())[:-1])
in_features = backbone.classifier[1].in_features

model = nn.Sequential(
    backbone_body,
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.21),
    nn.Linear(256, len(dataset.classes)),
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=max(len(train_loader),1), epochs=EPOCHS, pct_start=0.1)

print('Model ready on', device)

In [ ]:
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler(enabled=torch.cuda.is_available())
save_dir = Path('/content/checkpoints')
save_dir.mkdir(parents=True, exist_ok=True)
drive_dir = Path('/content/drive/MyDrive/deepfake_checkpoints') if USE_DRIVE else None
if drive_dir is not None:
    drive_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item() * images.size(0)

    epoch_loss = running / len(train_loader.dataset)
    print(f'Epoch {epoch}/{EPOCHS} loss: {epoch_loss:.4f}')

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'classes': dataset.classes,
        'config': {
            'subset_size': SUBSET_SIZE,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'lr': LR
        }
    }

    local_path = save_dir / f'rgb_finetune_epoch{epoch}.pth'
    torch.save(ckpt, local_path)
    print('Saved:', local_path)

    if drive_dir is not None:
        drive_path = drive_dir / local_path.name
        torch.save(ckpt, drive_path)
        print('Saved to Drive:', drive_path)

print('Finetune complete.')

## Next Steps
1. Download the best checkpoint from `/content/checkpoints` or Drive.
2. Place it in the repo as `results/checkpoints/fusion_best.pt` (or send it to me and I will wire it in).
3. Restart the interface and test on in-domain/out-of-domain examples.